In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
calfresh = pd.read_csv("calfresh.csv")

In [3]:
calfresh.head(5)

,Date,ZipCode,CalfreshDollars,meal_cost,CalfreshMeals
0,1/1/2020,91901,101567.0,3.473,29244.742330
1,1/1/2020,91902,88881.0,3.473,25591.992910
2,1/1/2020,91903,8938.0,3.473,2573.567271
3,1/1/2020,91905,37603.0,3.473,10827.237650
4,1/1/2020,91906,64151.0,3.473,18471.348620


In [4]:
calfresh.dtypes

Date                   str
ZipCode              int64
CalfreshDollars    float64
meal_cost          float64
CalfreshMeals      float64
dtype: object

In [5]:
calfresh_clean = calfresh.copy()

In [6]:
calfresh_clean.rename(columns={"ZipCode": "Zip Code",
                        "CalfreshDollars": "Calfresh Dollars",
                        "meal_cost": "Meal Cost",
                        "CalfreshMeals": "Calfresh Meals"},
               inplace=True)

**Convert Date from string to datetime**

In [7]:
from datetime import datetime

calfresh_clean["Date"] = pd.to_datetime(calfresh_clean["Date"], errors="coerce")

calfresh_clean["Month Year"] = (
    pd.to_datetime(calfresh_clean["Date"], errors="coerce")
    .dt.to_period("M")
    .astype(str)
)

In [8]:
calfresh_clean

,Date,Zip Code,Calfresh Dollars,Meal Cost,Calfresh Meals,Month Year
0,2020-01-01,91901,101567.0,3.473000,29244.742330,2020-01
1,2020-01-01,91902,88881.0,3.473000,25591.992910,2020-01
2,2020-01-01,91903,8938.0,3.473000,2573.567271,2020-01
3,2020-01-01,91905,37603.0,3.473000,10827.237650,2020-01
4,2020-01-01,91906,64151.0,3.473000,18471.348620,2020-01
...,...,...,...,...,...,...
10874,2025-08-01,92193,190982.0,4.657416,41006.000150,2025-08
10875,2025-08-01,92195,23563.0,4.657416,5059.243183,2025-08
10876,2025-08-01,92196,14233.0,4.657416,3055.986429,2025-08
10877,2025-08-01,92198,10415.0,4.657416,2236.218552,2025-08


**Set Zip Code to a string and zero pad it**

In [9]:
calfresh_clean["Zip Code"] = calfresh_clean["Zip Code"].astype(str).str.zfill(5)

In [10]:
def clean_zip(z):
    if pd.isna(z): return pd.NA
    z_str = str(z).strip()
    if z_str in ("", "nan", "00nan"): return pd.NA
    try: return str(int(float(z_str))).zfill(5)
    except ValueError: return pd.NA

calfresh_clean["Zip Code"] = calfresh_clean["Zip Code"].apply(clean_zip)

**Check for null values**

In [11]:
calfresh_clean.isna().sum()

Date                0
Zip Code            0
Calfresh Dollars    0
Meal Cost           0
Calfresh Meals      0
Month Year          0
dtype: int64

**Check for duplication**

In [12]:
dupe_check = calfresh_clean.groupby(["Zip Code", "Month Year"]).size()
print(f"{(dupe_check > 1).sum()} zip-months have more than one row")

0 zip-months have more than one row


**Filter for zip codes within the san diego region**

In [13]:
san_diego_zips_standardized = [
    "91901", "91902", "91903", "91905", "91906", "91908", "91909", "91910",
    "91911", "91912", "91913", "91914", "91915", "91916", "91917", "91921",
    "91931", "91932", "91933", "91934", "91935", "91941", "91942", "91943",
    "91944", "91945", "91946", "91950", "91951", "91962", "91963", "91976",
    "91977", "91978", "91979", "91980", "92003", "92004", "92007", "92008",
    "92009", "92010", "92011", "92013", "92014", "92018", "92019", "92020",
    "92021", "92022", "92023", "92024", "92025", "92026", "92027", "92028",
    "92029", "92030", "92033", "92036", "92037", "92038", "92039", "92040",
    "92046", "92049", "92051", "92052", "92054", "92055", "92056", "92057",
    "92058", "92059", "92060", "92061", "92064", "92065", "92066", "92067",
    "92068", "92069", "92070", "92071", "92072", "92074", "92075", "92078",
    "92079", "92081", "92082", "92083", "92084", "92085", "92086", "92088",
    "92091", "92092", "92093", "92101", "92102", "92103", "92104", "92105",
    "92106", "92107", "92108", "92109", "92110", "92111", "92112", "92113",
    "92114", "92115", "92116", "92117", "92118", "92119", "92120", "92121",
    "92122", "92123", "92124", "92126", "92127", "92128", "92129", "92130",
    "92131", "92136", "92137", "92138", "92139", "92140", "92142", "92143",
    "92145", "92149", "92150", "92153", "92154", "92159", "92160", "92163",
    "92165", "92166", "92167", "92168", "92169", "92170", "92171", "92172",
    "92173", "92174", "92175", "92176", "92177", "92178", "92186", "92191",
    "92192", "92193", "92195", "92196", "92198", 
]

sd_zip_strs = set(str(z).zfill(5) for z in san_diego_zips_standardized)

In [14]:
calfresh_clean["Zip_Code_In_San_Diego_County"] = calfresh_clean["Zip Code"].isin(sd_zip_strs)

In [15]:
calfresh_sd_records = (
    calfresh_clean[calfresh_clean["Zip_Code_In_San_Diego_County"]]
    .drop(columns="Zip_Code_In_San_Diego_County")
    .copy()
)

In [16]:
invalid_records = (
    calfresh_clean[~calfresh_clean["Zip_Code_In_San_Diego_County"]]
    .drop(columns=["Zip_Code_In_San_Diego_County"])
    .copy()
)

In [17]:
calfresh_sd_records

,Date,Zip Code,Calfresh Dollars,Meal Cost,Calfresh Meals,Month Year
0,2020-01-01,91901,101567.0,3.473000,29244.742330,2020-01
1,2020-01-01,91902,88881.0,3.473000,25591.992910,2020-01
2,2020-01-01,91903,8938.0,3.473000,2573.567271,2020-01
3,2020-01-01,91905,37603.0,3.473000,10827.237650,2020-01
4,2020-01-01,91906,64151.0,3.473000,18471.348620,2020-01
...,...,...,...,...,...,...
10873,2025-08-01,92192,11205.0,4.657416,2405.840507,2025-08
10874,2025-08-01,92193,190982.0,4.657416,41006.000150,2025-08
10875,2025-08-01,92195,23563.0,4.657416,5059.243183,2025-08
10876,2025-08-01,92196,14233.0,4.657416,3055.986429,2025-08


In [18]:
invalid_records

,Date,Zip Code,Calfresh Dollars,Meal Cost,Calfresh Meals,Month Year
10717,2025-07-01,92672,3419.0,4.725154,723.574339,2025-07
10878,2025-08-01,92672,3702.0,4.657416,794.861362,2025-08


In [19]:
calfresh_sd_records.to_csv("calfresh_standardized.csv", index=False)
invalid_records.to_csv("calfresh_invalid_zip_codes.csv", index=False)

In [20]:
print(f"{len(invalid_records)} of {len(calfresh_clean)} rows "
      f"({len(invalid_records)/len(calfresh_clean)*100:.1f}%) excluded for invalid zip codes")

2 of 10879 rows (0.0%) excluded for invalid zip codes


In [21]:
full_month_range = pd.period_range(
    start=calfresh_sd_records["Month Year"].min(),
    end=calfresh_sd_records["Month Year"].max(),
    freq="M"
).astype(str)

full_month_range

Index(['2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06',
       '2020-07', '2020-08', '2020-09', '2020-10', '2020-11', '2020-12',
       '2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06',
       '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12',
       '2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06',
       '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12',
       '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06',
       '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12',
       '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06',
       '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12',
       '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06',
       '2025-07', '2025-08'],
      dtype='str')

In [22]:
zip_month_present = (
    calfresh_sd_records.groupby("Zip Code")["Month Year"]
    .apply(lambda s: set(s.unique()))
)

missing_months_by_zip = {
    zip_code: sorted(set(full_month_range) - present_months)
    for zip_code, present_months in zip_month_present.items()
}

missing_months_df = pd.DataFrame([
    {"Zip Code": zip_code, "Missing Month": str(month)}
    for zip_code, months in missing_months_by_zip.items()
    for month in months
])

In [23]:
print(f"{missing_months_df['Zip Code'].nunique()} zip codes have at least one missing month")

5 zip codes have at least one missing month


In [24]:
missing_count_by_month = (
    missing_months_df.groupby("Missing Month")
    .size()
    .rename("Missing_Count")
    .reset_index()
    .sort_values("Missing Month")
)

missing_count_by_month

,Missing Month,Missing_Count
0,2022-06,2
1,2022-07,3
2,2022-08,3
3,2022-09,2
4,2022-10,2
5,2022-11,2
6,2022-12,3
7,2023-01,3
8,2023-02,3
9,2023-03,3


In [25]:
missing_count_by_zip_code = (
    missing_months_df.groupby("Zip Code")
    .size()
    .rename("Missing_Count")
    .reset_index()
    .sort_values("Missing_Count", ascending=False)
)

missing_count_by_zip_code

,Zip Code,Missing_Count
4,92191,39
3,92186,21
0,92039,9
2,92091,1
1,92060,1


**Check for outliers by Calfresh Dollars 	Meal Cost 	Calfresh Meals**

In [26]:
calfresh_sd_records

,Date,Zip Code,Calfresh Dollars,Meal Cost,Calfresh Meals,Month Year
0,2020-01-01,91901,101567.0,3.473000,29244.742330,2020-01
1,2020-01-01,91902,88881.0,3.473000,25591.992910,2020-01
2,2020-01-01,91903,8938.0,3.473000,2573.567271,2020-01
3,2020-01-01,91905,37603.0,3.473000,10827.237650,2020-01
4,2020-01-01,91906,64151.0,3.473000,18471.348620,2020-01
...,...,...,...,...,...,...
10873,2025-08-01,92192,11205.0,4.657416,2405.840507,2025-08
10874,2025-08-01,92193,190982.0,4.657416,41006.000150,2025-08
10875,2025-08-01,92195,23563.0,4.657416,5059.243183,2025-08
10876,2025-08-01,92196,14233.0,4.657416,3055.986429,2025-08


In [27]:
negative = calfresh_sd_records[
    (calfresh_sd_records["Calfresh Dollars"] < 0) | 
    (calfresh_sd_records["Meal Cost"] < 0) | 
    (calfresh_sd_records["Calfresh Meals"] < 0)
]

In [28]:
negative

,Date,Zip Code,Calfresh Dollars,Meal Cost,Calfresh Meals,Month Year
4692,2022-06-01,91943,-6076.0,4.36032,-1393.475824,2022-06
4731,2022-06-01,92039,-11806.0,4.36032,-2707.599667,2022-06
4742,2022-06-01,92060,-7527.0,4.36032,-1726.249593,2022-06
4764,2022-06-01,92091,-4341.0,4.36032,-995.569215,2022-06


In [29]:
calfresh_sd_records = calfresh_sd_records[
    (calfresh_sd_records["Calfresh Dollars"] >= 0) & 
    (calfresh_sd_records["Meal Cost"] >= 0) & 
    (calfresh_sd_records["Calfresh Meals"] >= 0)
]

In [30]:
def flag_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    is_outlier = (df[column] < lower_bound) | (df[column] > upper_bound)
    return is_outlier, lower_bound, upper_bound

for col in ["Calfresh Dollars", "Meal Cost", "Calfresh Meals"]:
    outlier_flag, lower, upper = flag_outliers(calfresh_sd_records, col)
    calfresh_sd_records[f"{col}_Outlier"] = outlier_flag
    n_outliers = outlier_flag.sum()
    print(f"{col}: bounds [{lower:.2f}, {upper:.2f}] — "
          f"{n_outliers} outliers ({n_outliers/len(calfresh_sd_records)*100:.1f}%)")

calfresh_sd_records["Any_Outlier"] = (
    calfresh_sd_records["Calfresh Dollars_Outlier"] |
    calfresh_sd_records["Meal Cost_Outlier"] |
    calfresh_sd_records["Calfresh Meals_Outlier"]
)
print(f"\n{calfresh_sd_records['Any_Outlier'].sum()} of {len(calfresh_sd_records)} rows "
      f"flagged as an outlier on at least one column")


Calfresh Dollars: bounds [-907045.00, 1569891.00] — 911 outliers (8.4%)
Meal Cost: bounds [2.75, 5.55] — 0 outliers (0.0%)
Calfresh Meals: bounds [-216700.88, 375133.79] — 892 outliers (8.2%)

941 of 10873 rows flagged as an outlier on at least one column


In [31]:
for col in ["Calfresh Dollars", "Calfresh Meals", "Meal Cost"]:
    low, up = flag_outliers(calfresh_sd_records, col)[1:]
    n_low = (calfresh_sd_records[col] < low).sum()
    n_high = (calfresh_sd_records[col] > up).sum()
    print(f"{col}: {n_low} below lower bound, {n_high} above upper bound")

Calfresh Dollars: 0 below lower bound, 911 above upper bound
Calfresh Meals: 0 below lower bound, 892 above upper bound
Meal Cost: 0 below lower bound, 0 above upper bound


In [32]:
outlier_records = calfresh_sd_records[calfresh_sd_records["Any_Outlier"]]

zip_summary_short = (
    outlier_records.groupby("Zip Code")
    .size()
    .rename("Times_Flagged")
    .reset_index()
)
zip_summary_short["Pct_of_Total_Outliers"] = (
    zip_summary_short["Times_Flagged"] / zip_summary_short["Times_Flagged"].sum() * 100
).round(1)
zip_summary_short = zip_summary_short.sort_values("Times_Flagged", ascending=False)

zip_summary_short

,Zip Code,Times_Flagged,Pct_of_Total_Outliers
11,92105,68,7.2
5,92020,68,7.2
6,92021,68,7.2
2,91950,65,6.9
1,91911,65,6.9
0,91910,65,6.9
3,91977,65,6.9
15,92115,65,6.9
14,92114,65,6.9
13,92113,65,6.9


In [33]:
negative.to_csv("calfresh_negative_values.csv", index=False)

In [34]:
calfresh_sd_records.to_csv("CalFresh_cleaned.csv", index=False)